# 03 機械学習モデル & 市場レジーム分析

## 目的
- LightGBM / XGBoost / RandomForest で出来高シグナルの予測力を検証
- Walk-Forward CVで時系列に忠実な評価
- 市場レジーム（上昇/下落/横ばい）別の戦略パフォーマンスを比較

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)
%matplotlib inline

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s')

In [ ]:
df = pd.read_pickle('../quant_research/data/_intermediate_df_features.pkl')
print(f"データ: {df['Code'].nunique():,} 銘柄, {len(df):,} 行")

## STEP 6: 機械学習モデル

目的変数: 5日後リターン > 3%

In [ ]:
from quant_research.ml_models import run_walk_forward_cv, FEATURE_COLUMNS

features = [f for f in FEATURE_COLUMNS if f in df.columns]
print(f"使用特徴量: {len(features)}")
for f in features:
    print(f"  {f}")

In [ ]:
# Walk-Forward CV実行（ハイパーパラメータ最適化あり）
ml_results = run_walk_forward_cv(df, features=features, optimize=True)

print("\n=== モデル比較 ===")
for name, res in ml_results.items():
    avg = res.get('metrics_avg', {})
    print(f"\n{name}:")
    print(f"  AUC:       {avg.get('roc_auc', 0):.4f}")
    print(f"  Precision: {avg.get('precision', 0):.4f}")
    print(f"  Recall:    {avg.get('recall', 0):.4f}")
    print(f"  F1:        {avg.get('f1', 0):.4f}")

In [ ]:
# 特徴量重要度
from quant_research.reporter import plot_feature_importance

plot_feature_importance(
    ml_results, 
    save_path='../quant_research/data/reports/feature_importance.png'
)
plt.show()

# 各モデルのTop 5特徴量
for name, res in ml_results.items():
    fi = res.get('feature_importance')
    if fi is not None:
        print(f"\n{name} Top 5 Features:")
        for feat, imp in fi.head(5).items():
            print(f"  {feat}: {imp:.4f}")

## STEP 7: 市場レジーム分析

In [ ]:
from quant_research.data_fetcher import _load_cache
from quant_research.regime_analyzer import (
    classify_regime, merge_regime, backtest_by_regime,
    regime_performance_summary, get_current_regime,
    regime_transition_matrix,
)

# 指数データ読み込み
topix = _load_cache('index_topix')
if topix is None or topix.empty:
    topix = _load_cache('index_nikkei')

if topix is not None and not topix.empty:
    regime_df = classify_regime(topix)
    
    # レジーム推移の可視化
    fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
    
    axes[0].plot(regime_df['Date'], regime_df['Close_idx'], color='black', linewidth=0.8)
    axes[0].plot(regime_df['Date'], regime_df['ma_short'], color='orange', 
                 linewidth=0.5, alpha=0.7, label='50MA')
    axes[0].plot(regime_df['Date'], regime_df['ma_long'], color='blue', 
                 linewidth=0.5, alpha=0.7, label='200MA')
    axes[0].set_title('Index Price with Moving Averages')
    axes[0].legend()
    
    colors = {'bull': '#2ecc71', 'bear': '#e74c3c', 'sideways': '#f39c12'}
    for regime, color in colors.items():
        mask = regime_df['regime'] == regime
        axes[1].fill_between(
            regime_df['Date'], 0, 1,
            where=mask, color=color, alpha=0.5, label=regime
        )
    axes[1].set_title('Market Regime')
    axes[1].legend()
    axes[1].set_yticks([])
    
    plt.tight_layout()
    plt.savefig('../quant_research/data/reports/regime_timeline.png', dpi=150)
    plt.show()
    
    # 現在のレジーム
    current = get_current_regime(topix)
    print(f"\n現在の市場レジーム: {current}")
    
    # 遷移行列
    print(f"\nレジーム遷移確率:")
    print(regime_transition_matrix(regime_df).round(3))
else:
    print("指数データが利用できません")

In [ ]:
# レジーム別バックテスト
if topix is not None and not topix.empty:
    df_regime = merge_regime(df, regime_df)
    
    # 最適化結果から上位条件を取得
    opt = pd.read_pickle('../quant_research/data/_results_optimization_results.pkl')
    top_conditions = [r.condition for r in opt['all'][:20]] if opt and 'all' in opt else []
    
    if top_conditions:
        regime_bt = backtest_by_regime(df_regime, top_conditions)
        regime_summary = regime_performance_summary(regime_bt)
        
        print("\n=== レジーム別パフォーマンス ===")
        for _, row in regime_summary.iterrows():
            print(f"  {row['regime']:8s}: Best WR={row['best_win_rate']:.1%}, "
                  f"Best SR={row['best_sharpe']:.2f}, "
                  f"Best Ret={row['best_avg_return']:.2%}, "
                  f"N conditions={row['n_conditions']}")
        
        from quant_research.reporter import plot_regime_performance
        plot_regime_performance(
            regime_summary,
            save_path='../quant_research/data/reports/regime_performance.png'
        )
        plt.show()
        
        pd.to_pickle(regime_bt, '../quant_research/data/_results_regime_results.pkl')
        regime_summary.to_pickle('../quant_research/data/_intermediate_regime_summary.pkl')
        pd.to_pickle(current, '../quant_research/data/_results_current_regime.pkl')

In [ ]:
# ML結果保存
pd.to_pickle(ml_results, '../quant_research/data/_results_ml_results.pkl')
print("保存完了")